# Regularization Experiments
**Hypothesis:** Dropout prevents overfitting | **Result:** Dropout hurts minority class

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping
from utils.training import train_two_stage
from utils.reproducibility_utils import seed_everything
from models.vanilla_rnn import VanillaRNN
from models.simple_lstm import SimpleLSTM

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()
print(f"Train: {len(train_loader)} batches")

## 1. RNN Dropout Sweep

In [ ]:
dropout_results = []
for dropout in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    seed_everything(SEED)
    print(f"Training RNN dropout={dropout}...")
    model = VanillaRNN(22, 64, 1, dropout)
    _, metrics = train_two_stage(model, train_loader, val_loader, band_dict, device)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    dropout_results.append({'dropout': dropout, **metrics})

pd.DataFrame(dropout_results)

## 2. LSTM Dropout Comparison

In [ ]:
lstm_results = []
for dropout in [0.0, 0.2, 0.4]:
    seed_everything(SEED)
    print(f"Training LSTM dropout={dropout}...")
    model = SimpleLSTM(22, 64, 1, dropout)
    _, metrics = train_two_stage(model, train_loader, val_loader, band_dict, device)
    print(f"  Overall={metrics['overall_mae']:.4f}")
    lstm_results.append({'dropout': dropout, **metrics})

pd.DataFrame(lstm_results)

## Visualization

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.DataFrame(dropout_results)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Overall MAE vs Dropout', 'Few-shot MAE vs Dropout'])
fig.add_trace(go.Scatter(x=df['dropout'], y=df['overall_mae'], mode='lines+markers', name='Overall'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['dropout'], y=df['few_mae'], mode='lines+markers', name='Few-shot', line=dict(color='red')), row=1, col=2)
fig.update_layout(title='Dropout Impact on Performance', width=1000, height=400)
fig.show()

## Summary

| Issue | Explanation |
|-------|-------------|
| Signal attenuation | Few-shot (5%) × (1-p)³ ≈ 1.7% effective signal |
| Data scarcity | Stochastic masking worsens minority class learning |

**Conclusion:** Constrain model capacity instead of using dropout for imbalanced regression.